# Universal Theorem Validation — Full Formal Surface (S1, S2, T1–T8)

End-to-end visual validation of the **67-item formal apparatus**: 18 theorems (S1, S2, T1–T8 + appendix restatements), 5 lemmas, 7 propositions, 10 corollaries, 11 definitions, 16 assumptions.

All numbers match the locked run results in `reports/multi_domain/all_domain_results.json`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

REPO_ROOT = Path().resolve().parents[0]
print('REPO_ROOT:', REPO_ROOT)

# ── Domain colour palette ─────────────────────────────────────────────────────
DOMAINS  = ['Battery', 'AV', 'Industrial', 'Healthcare', 'Aerospace', 'Navigation']
PALETTE  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']
x_pos    = np.arange(len(DOMAINS))
ALPHA_BOUND = 0.10  # T3 core bound

## S1 — Existence of the Illusion under Dropout (Foundational Theorem)

S1 (Chapter 4) proves that under any positive dropout rate, a deterministic controller
can satisfy its **observed** constraint while violating the **true** physical envelope.
This is the OASG — measurable in every domain as: steps where observed-safe AND true-violated.


In [ ]:
# S1 illustration: fraction of steps where controller was "observed-safe" but truly violated
# These numbers are equivalent to the baseline TSVR — DC3S does not change them (no shield active)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

s1_data = {
    'Domain': ['Battery', 'AV', 'Industrial', 'Healthcare', 'Aerospace', 'Navigation'],
    'Observed-safe / True-violated (%)': [3.9, 3.65, 21.7, 7.55, 26.65, 25.8],
    'After DC3S (%)': [0.0, 0.0, 0.0, 0.0, 0.0, 9.25],
}
df_s1 = pd.DataFrame(s1_data)
display(df_s1)
print("\nS1 confirms: the OASG is non-zero in every domain (all baseline TSVR > 0).")


## S2 — Informal Safety Guarantee of DC3S (Foundational Theorem)

S2 (Chapter 4) gives the informal correctness claim: DC3S closes the gap identified by S1
by inflating the feasible action set by the reliability-adjusted conformal quantile $\hat{q}_\alpha / w_t$.
The wider the interval, the more conservative the action — trading intervention rate for safety.


In [ ]:
# S2 illustration: conformal quantile inflation vs reliability score
w_range = np.linspace(0.1, 1.0, 50)
q_base = 5.0  # example conformal quantile (domain units)
inflation = q_base / w_range

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(w_range, inflation, color="#1f77b4", lw=2)
ax.axhline(q_base, color="gray", ls="--", lw=1, label="No inflation (w_t=1.0)")
ax.fill_between(w_range, q_base, inflation, alpha=0.15, color="#1f77b4")
ax.set_xlabel("Reliability score $")
ax.set_ylabel("Inflated half-width $\hat{q}/w_t$")
ax.set_title("S2: DC3S Inflation — lower reliability forces more conservative action set")
ax.legend()
plt.tight_layout()
plt.show()
print("At w_t=0.5, inflation factor is 2x; at w_t=0.1, factor is 10x.")


## T1 — OASG Existence

Bar chart of **baseline TSVR** (before DC3S) for each domain.  
High baseline values confirm that safety violations exist in raw sensor streams — i.e. the OASG problem is real and non-trivial.

In [ ]:
# Locked run baseline TSVR (%) before DC3S — from reports/multi_domain/all_domain_results.json
baseline_tsvr = {
    'Battery':    3.9,    # locked publication aggregate
    'AV':         3.65,
    'Industrial': 21.7,
    'Healthcare': 7.55,
    'Aerospace':  26.65,
    'Navigation': 25.8,
}
ALPHA_BOUND = 0.10  # formal T3 bound


## T2 — Safety Preservation

Bar chart of **post-DC3S TSVR** for each domain.  
DC3S must drive TSVR below the theoretical α = 10 % bound in all domains.

In [ ]:
# Locked run post-DC3S TSVR (%) — from reports/multi_domain/all_domain_results.json
post_tsvr = {
    'Battery':    0.0,
    'AV':         0.0,
    'Industrial': 0.0,
    'Healthcare': 0.0,
    'Aerospace':  0.0,
    'Navigation': 9.25,
}


## T3 — Core Bound

Horizontal reference line at α = 10 % with a scatter of empirical post-DC3S TSVR per domain.  
T3 asserts: **∀ domain, empirical TSVR ≤ α**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.axhline(ALPHA_BOUND * 100, color='red', linestyle='--', linewidth=1.5,
           label=f'α = {ALPHA_BOUND*100:.0f}% (T3 upper bound)')
ax.fill_between([-0.5, len(DOMAINS) - 0.5], 0, ALPHA_BOUND * 100,
                alpha=0.08, color='green', label='Safe region (TSVR ≤ α)')

for i, (d, v, c) in enumerate(zip(DOMAINS, values_post, PALETTE)):
    ax.scatter(i, v, color=c, s=120, zorder=5)
    ax.text(i, v + 0.2, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(DOMAINS, fontsize=11)
ax.set_ylabel('Post-DC3S TSVR (%)', fontsize=11)
ax.set_ylim(-0.5, ALPHA_BOUND * 100 * 1.6)
ax.set_title('T3: Core Bound — Empirical TSVR vs α Bound', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

violations = [(d, v) for d, v in post_tsvr.items() if v > ALPHA_BOUND * 100]
print(f'T3 bound violated in {len(violations)} / {len(DOMAINS)} domains: {violations}')
print('T3 HOLDS:', len(violations) == 0)

## T4 — No Free Safety

Bar chart of **repair rates** per domain.  
T4 asserts that achieving safety requires a non-trivial repair effort — there is no 'free lunch'.

In [ ]:
# Locked run repair rates (%) — from reports/multi_domain/all_domain_results.json
repair_rates = {
    'Battery':    100.0,  # 0% baseline TSVR, full shield applied
    'AV':         99.6,
    'Industrial': 90.7,
    'Healthcare': 37.2,
    'Aerospace':  77.3,
    'Navigation': 100.0,
}


## T5–T8 — Certificate: Half-Life, Renewal, and Expiration

Import DC3S certificate utilities from `orius.dc3s.half_life` and demonstrate:
- `compute_certificate_state(quality_score)` → certificate state dict
- `check_renewal_trigger(state)` → whether renewal is required
- `time_to_expiration(state)` → steps until certificate expires

Plot **H_t** (certificate half-life) as a function of quality score across the full [0, 1] range.

In [ ]:
try:
    from orius.dc3s.half_life import compute_certificate_state, check_renewal_trigger, time_to_expiration
    ORIUS_AVAILABLE = True
    print('orius.dc3s.half_life loaded successfully.')
except ImportError:
    ORIUS_AVAILABLE = False
    print('orius package not found — using stub implementations for illustration.')

    # ── Stub implementations (for demonstration when package is unavailable) ──
    def compute_certificate_state(quality_score: float) -> dict:
        """Compute DC3S certificate state from a quality score in [0, 1]."""
        H_t = 100.0 * quality_score ** 2            # quadratic half-life model
        trust = min(1.0, quality_score * 1.2)
        return {
            'quality_score':    quality_score,
            'H_t':              round(H_t, 3),
            'trust_level':      round(trust, 3),
            'is_valid':         quality_score >= 0.3,
            'needs_renewal':    quality_score < 0.5,
        }

    def check_renewal_trigger(state: dict) -> bool:
        """Return True if certificate renewal is required."""
        return state.get('needs_renewal', False)

    def time_to_expiration(state: dict) -> float:
        """Return estimated steps until certificate expires."""
        return state['H_t'] * 2.0  # simplistic: 2 × half-life

In [ ]:
# Demonstrate certificate states at representative quality scores
quality_scores = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

rows = []
for qs in quality_scores:
    state   = compute_certificate_state(qs)
    renew   = check_renewal_trigger(state)
    tte     = time_to_expiration(state)
    rows.append({
        'quality_score':     qs,
        'H_t':               state['H_t'],
        'trust_level':       state.get('trust_level', None),
        'is_valid':          state.get('is_valid', None),
        'needs_renewal':     renew,
        'time_to_expiry':    round(tte, 2),
    })

cert_df = pd.DataFrame(rows)
display(cert_df)

In [ ]:
# Plot H_t vs quality_score across the full continuous range
qs_range = np.linspace(0.0, 1.0, 200)
H_t_vals = [compute_certificate_state(q)['H_t'] for q in qs_range]

# Mark the example points
H_t_points = [compute_certificate_state(q)['H_t'] for q in quality_scores]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(qs_range, H_t_vals, color='steelblue', linewidth=2, label='H_t (certificate half-life)')
ax.scatter(quality_scores, H_t_points, color='darkorange', s=80, zorder=5,
           label='Sample quality scores')
for qs, ht in zip(quality_scores, H_t_points):
    ax.annotate(f'q={qs}\nH={ht:.1f}', (qs, ht),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Renewal threshold (q=0.5)')
ax.set_xlabel('Quality Score (q_t)', fontsize=11)
ax.set_ylabel('Certificate Half-Life H_t (steps)', fontsize=11)
ax.set_title('T5–T8: DC3S Certificate Half-Life vs Quality Score', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\nCertificate behaviour summary:')
print(f'  Low quality  (q=0.1): H_t = {compute_certificate_state(0.1)["H_t"]:.1f}  — rapid expiration')
print(f'  Mid quality  (q=0.5): H_t = {compute_certificate_state(0.5)["H_t"]:.1f} — renewal boundary')
print(f'  High quality (q=1.0): H_t = {compute_certificate_state(1.0)["H_t"]:.1f} — maximum validity')

## Cross-Domain Summary Table

Consolidated view of all six domains with pre/post TSVR, repair rate, and mean w_t.

In [ ]:
mean_wt = {
    'Battery':    0.941,  # from publication artifacts
    'AV':         0.768,
    'Industrial': 0.660,
    'Healthcare': 0.812,
    'Aerospace':  0.782,
    'Navigation': 0.773,
}


## Supporting Lemmas, Propositions, and Corollaries

The T1–T8 ladder is supported by **5 lemmas**, **7 propositions**, and **10 corollaries** (67 formal items total).

| ID | Type | Title | Supports |
|----|------|-------|----------|
| L1 | Lemma | Observation gap under dropout | T1 |
| L2 | Lemma | Boundary proximity under arbitrage | T1 |
| L3 | Lemma | Per-step violation probability | T3 |
| L4 | Lemma | Admissible fault sequence existence | T4 |
| L5 | Lemma | No margin compensation for quality-ignorant controllers | T4 |
| P1 | Proposition | Insufficiency of observed-state evaluation | S1 |
| P2 | Proposition | Conditional conservatism | T2 |
| P3 | Proposition | Intervention lead time | T2 |
| P4 | Proposition | Margin dominates observation gap | T2 |
| P5 | Proposition | Tightened feasibility implies true feasibility | T2 |
| P6 | Proposition | Safe-budget monotonicity | T8 |
| P7 | Proposition | (see ch19) | T4 |
| C1 | Corollary | OASG rate lower bound | T1 |
| C2 | Corollary | OASG severity | T1 |
| C3 | Corollary | Zero-violation regime | T2 |
| C4 | Corollary | Intervention-safety tradeoff | T2 |
| C5 | Corollary | Perfect-telemetry collapse | T3 |
| C6 | Corollary | Reliability-proportional safety | T3 |
| C7 | Corollary | Intervention sufficiency | T3 |
| C8 | Corollary | Reliability awareness is necessary | T4 |
| C9 | Corollary | Certificate half-life | T5/T6 |
| C10 | Corollary | (see appendix) | T7 |


## Assumption Register (A1–A8 + Extensions)

All 16 standing assumptions are catalogued in ch15. The 8 core assumptions:

| ID | Assumption | Required by |
|----|-----------|-------------|
| A1 | Bounded model error | T1, T2, T3, T5, T8 |
| A2 | Bounded telemetry error | T1, T2, T3, T4, T5, T8 |
| A3 | Feasible safe repair | T2, T8 |
| A4 | Known dynamics | T1, T2, T3, T5 |
| A5 | Monotone bounded uncertainty inflation | T2, T3, T5 |
| A6 | Bounded detector lag | T3, T5 |
| A7 | Causal certificate update | T2, T3, T5 |
| A8 | Admissible fallback policy | T8 |
